# Solar Influence

Detect whether a room temperature sensor is influenced by **direct solar radiation**, using the room temperature, the outdoor temperature and the global radiation.

This example builds a small synthetic, sun-exposed dataset so it runs without external files.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Build a synthetic sun-exposed sensor

In [ ]:
import numpy as np
import pandas as pd

idx = pd.date_range("2024-06-01", periods=24 * 14, freq="h")
hour = idx.hour
solar = np.clip(800 * np.sin((hour - 6) / 12 * np.pi), 0, None)
t_out = 18 + 5 * np.sin(np.arange(len(idx)) / 24)
rng = np.random.default_rng(0)
t_room = t_out + 0.01 * solar + rng.normal(0, 0.1, len(idx))

mk = lambda v: pd.DataFrame({"timestamp": idx, "value": v})
room, outdoor, radiation = mk(t_room), mk(t_out), mk(solar)

## Analyze solar influence

`analyze_solar_influence` returns the Pearson correlation between radiation and excess temperature, the local peak hour (orientation hint), and the steep-rise event rate.

In [ ]:
from pyedautils.data_prep.solar_influence import analyze_solar_influence

analyze_solar_influence(room, outdoor, radiation, local_tz="Europe/Zurich")

## Dual-axis plot

`plot_solar_influence` overlays room temperature (left axis) and global radiation (right axis); flagged steep-rise events are marked.

In [ ]:
from pyedautils.plots import plot_solar_influence

joined = pd.DataFrame({"t_room": t_room, "solar": solar}, index=idx)
joined["is_event"] = joined["solar"] > 700
fig = plot_solar_influence(joined, event_col="is_event")
fig.show()